In [1]:
import os
import numpy as np
import pandas as pd

import torch

from torchvision.transforms import v2
import torchvision.transforms as transformers
import torchvision.transforms.functional as F

from PIL import Image
from tqdm import tqdm

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

In [2]:
from data_utilities.training_dataloader import HouseTrainingDataset

## Compute Device Selection

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("using device:", device)

using device: mps


## Loading Training Data and Creating Training Dataloader

In [4]:
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [5]:
class LetterboxResize(torch.nn.Module):
    """
    Resize preserving aspect ratio with the longer side defining the size,
    then pad the short side to a square using grey 114 value
    """
    def __init__(self, size=224, fill=114):
        super().__init__()
        self.size = size
        self.fill = fill

    def forward(self, img):
        _, h, w = img.shape
        scale = self.size / max(h, w)
        new_h, new_w = round(h * scale), round(w * scale)
        img = F.resize(img, [new_h, new_w], interpolation=F.InterpolationMode.BICUBIC)

        pad_h, pad_w = self.size - new_h, self.size - new_w
        top, bottom = pad_h // 2, pad_h - pad_h // 2
        left, right = pad_w // 2, pad_w - pad_w // 2
        return F.pad(img, [left, top, right, bottom], fill=self.fill)

In [6]:
training_dataset = HouseTrainingDataset("house_dataset/train", "house_dataset/train.csv")

image_transformation = v2.Compose([
    LetterboxResize(IMG_SIZE),
    v2.ToDtype(torch.float32, scale=True),          # uint8 [0,255] -> float32 [0,1]
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

training_dataset.set_transform(image_transformation)

train_dataloader = training_dataset.get_dataloader(batch_size=16, shuffle=False)

## DinoV3 Model Loading

In [7]:
MODEL_NAME = "dinov3_vith16plus"
weights_directory = "dinov3_ViT_weights"
weights_filename = "dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth"
WEIGHTS = os.path.join(weights_directory, weights_filename)

model = torch.hub.load(
    "facebookresearch/dinov3", MODEL_NAME, source="github", 
    weights=WEIGHTS
)

Using cache found in /Users/act133/.cache/torch/hub/facebookresearch_dinov3_main


In [8]:
model = model.to(device)

## Obtain Vector Embeddings from DinoV3

In [9]:
all_embeddings, all_prices, all_filenames = [], [], []
 
with torch.no_grad():
    for imgs, prices, fnames in tqdm(train_dataloader):
        imgs = imgs.to(device)
        features = model.forward_features(imgs)
        cls = features["x_norm_clstoken"]                     # (B, 768)
        patch_mean = features["x_norm_patchtokens"].mean(1)   # (B, 768)
        embedding = torch.cat([cls, patch_mean], dim=1)       # (B, 1536)
 
        all_embeddings.append(embedding.cpu().numpy())
        all_prices.append(prices.numpy())
        all_filenames.extend(fnames)
 
X = np.concatenate(all_embeddings)
y = np.concatenate(all_prices)
 
np.save("train_embeddings.npy", X)
np.save("train_prices.npy", y)
print("saved embeddings:", X.shape, "prices:", y.shape)

100%|████████████████████████████████████████████████████████████████████████████████| 500/500 [17:03<00:00,  2.05s/it]

saved embeddings: (8000, 2560) prices: (8000,)
